# B1b · Maskeyle daha fazla soru sormak

B1'de öğrendiğin şey: **maske = her satıra True/False**, `tablo[maske]` = True olan satırlar, `maske.sum()` = kaç tane.

Bugün bu tek fikri 5 yeni parçayla büyütüyoruz. Her parçada önce oyuncak tabloda **ben** gösteriyorum, sonra gerçek veride **sen** yazıyorsun.

**Kural aynı:** kod hücresinden önce tahminini yaz, sonra çalıştır.

In [1]:
import pandas as pd

oyuncak = pd.DataFrame({
    "saat":  [9, 12, 15, 18, 21],
    "gfs":   [4, 8, 12, 10, 3],
    "olcum": [6, 14, 20, 16, 5],
})
kuvvetli = oyuncak["olcum"] >= 15
oyuncak

,saat,gfs,olcum
0,9,4,6
1,12,8,14
2,15,12,20
3,18,10,16
4,21,3,5


## 1. Tek sütun mu, tablo mu?

B1'de `oyuncak["olcum"]` için "olcum tablosu" demiştin. Aslında tablo değil, **tek bir sütun**. Pandas'ta buna **Series** deniyor: başında satır numarası olan tek bir liste.

- `oyuncak["olcum"]`: **tek köşeli parantez** → Series (tek sütun)
- `oyuncak[["olcum", "gfs"]]`: **çift köşeli parantez**, içinde liste → yine tablo (DataFrame), yalnız o sütunlarla

`.shape` (satır sayısı, sütun sayısı) farkı gösterir. Series'in sütun sayısı yoktur, yalnız uzunluğu vardır.

In [2]:
print(oyuncak["olcum"].shape)
print(oyuncak[["olcum", "gfs"]].shape)
print(oyuncak[["olcum"]].shape)

(5,)
(5, 2)
(5, 1)


> Üçüncü satır neden `(5, 1)` çıktı da birincisi `(5,)` çıktı? (tek cümle)
>
> *direkt ölçüm diyince tablo değil sütun oluyo ve tek sutün olduğu için ,1 yazmıyor diğeri tek sütunlu bir tablo*

## 2. Satır **ve** sütun birlikte seçmek: `.loc`

`oyuncak[kuvvetli]` kuvvetli satırları **bütün sütunlarıyla** getiriyordu. Çoğu zaman yalnız bir sütuna bakmak isteriz, örneğin "kuvvetli saatlerde GFS ne demiş?"

```python
tablo.loc[satır_maskesi, "sütun"]
```

Virgülün solu **hangi satırlar**, sağı **hangi sütun**. Sonuna `.mean()` koyunca o seçimin ortalamasını alırsın.

In [3]:
oyuncak.loc[kuvvetli, "gfs"]

2    12
3    10
Name: gfs, dtype: int64

In [4]:
oyuncak.loc[kuvvetli, "gfs"].mean()

np.float64(11.0)

> Bu 11 sayısı oyuncak tablosunda nereden geldi? (tek cümle)
>
> *kuvvetli olanların gfssini döndürdü ve ortalamasını aldı*

## 3. "Kaç tane" yerine "yüzde kaç": maskenin `.mean()`'i

`.sum()` True'ları sayıyordu. `.mean()` ise True'ların **oranını** verir, çünkü True=1, False=0 sayılır, ortalama da "1'lerin payı" olur.

Oyuncakta 5 satırın 2'si kuvvetli → 2/5 = 0,4. Yüzde için 100'le çarparsın.

In [5]:
kuvvetli.mean()

np.float64(0.4)

## 4. "Veya" ve "değil": `|` ile `~`

B1'de `&` ("ve") kullandın. İki kardeşi daha var:

| İşaret | Anlamı | Ne zaman True |
|---|---|---|
| `&` | ve | iki koşul da doğruysa |
| `\|` | veya | en az biri doğruysa |
| `~` | değil | koşulu ters çevirir |

Yine her koşul kendi parantezinde.

In [6]:
# ölçüm 15+ VEYA gfs 10+ olan satırlar
oyuncak[(oyuncak["olcum"] >= 15) | (oyuncak["gfs"] >= 10)]

,saat,gfs,olcum
2,15,12,20
3,18,10,16


> Hangi satırlar geldi? `&` olsaydı hangileri gelirdi? (tahmin et, sonra istersen değiştirip dene)
>
> *2 ve 3 geldi ve olsaydı da aynıları gelirdi*

In [7]:
# kuvvetli OLMAYAN satırlar
oyuncak[~kuvvetli]

,saat,gfs,olcum
0,9,4,6
1,12,8,14
4,21,3,5


## 5. Tarihten soru sormak: `.dt` ve `.isin`

Gerçek veride `time` sütunu tarih-saat tipinde. `.dt` ile içinden parça çekilir:

- `veri["time"].dt.month` → ay (1–12)
- `veri["time"].dt.hour` → saat (0–23)

"Ay 6, 7 ya da 8 mi?" sorusunu üç ayrı `|` ile yazmak yerine `.isin([6, 7, 8])` yazılır.

⚠️ Saatler **UTC**. İzmir yerel saati UTC+3, yani UTC 12:00 = yerel 15:00.

In [8]:
from ruzgar.data import load_aligned
veri = load_aligned()

ay = veri["time"].dt.month
ay.head()

0    3
1    3
2    3
3    3
4    3
Name: time, dtype: int32

## 6. Boş değerler: `.isna()` ve `.notna()`

Gerçek verilerde boş hücreler olur, pandas bunları `NaN` olarak gösterir. `.isna()` "bu hücre boş mu?" diye soran bir **maskedir**, yani şimdiye kadar öğrendiğin her şeyle birlikte kullanılabilir.

In [9]:
veri.isna().sum()

time                0
speed_kn_gfs        0
dir_deg_gfs         0
gust_kn_gfs         0
station             0
speed_kn_obs        0
dir_deg_obs      3399
gust_kn_obs     23040
hata                0
dtype: int64

> Hangi iki sütunda boşluk var? `gust_kn_obs` neredeyse tamamen boş; tahminin, neden? (tek cümle)
>
> *gust neydi unuttum

---
## Senin sıran

Her alıştırmada **önce tahminini yaz**, sonra kodu yaz. Değişken adlarını sen seç.

**Alıştırma 4 (`.loc` + `.mean()`):** gerçek ölçümün 20+ knot olduğu saatlerde:
- ölçümün ortalaması kaç?
- **aynı saatlerde** GFS'in tahmininin ortalaması kaç?

Tahminin: GFS ortalaması ölçümün yaklaşık yüzde kaçı çıkar?

> Tahminim:
>
> *%60 tahminim*

In [10]:
kuvvetli = veri["speed_kn_obs"]>=20
ort_obs= veri.loc[kuvvetli,"speed_kn_obs"].mean()
ort_gfs = veri.loc[kuvvetli, "speed_kn_gfs"].mean()
print((ort_gfs/ort_obs)*100)

50.389153754469596


> İki ortalamayı karşılaştırınca GFS hakkında ne söylersin? (tek cümle)
>
> *50.389153754469596si çıktı çok yüksek bir hata payı var*

**Alıştırma 5 (maskenin `.mean()`'i):** bütün saatlerin **yüzde kaçında** gerçekte 20+ knot esmiş?

In [11]:
kuvvetli.mean()*100
# %3.36

np.float64(3.3670464729228007)

**Alıştırma 6 (`.dt.month`, `.isin`, `~`):** 20+ knot saatlerinin oranı **yazın** (Haziran, Temmuz, Ağustos) mı yüksek, yoksa yılın geri kalanında mı? İkisinin oranını ayrı ayrı bul.

İpucu: `yaz` diye bir maske yap. "Yazın kuvvetli oranı" = yaz satırlarını seç, sonra o satırlarda kuvvetli maskesinin ortalamasını al.

> Tahminim:
>
> *bence yılın geri kalanında daha yüksektir*

In [12]:
yaz = veri["time"].dt.month.isin([6,7,8])
print(yaz[kuvvetli].mean())
print(yaz[~kuvvetli].mean())


0.5310519645120405
0.2739357004062886


> Sonuç sörfçü olarak sana mantıklı geliyor mu? Neden? (tek cümle)
>
> yazın daha çokmuş uvvetli rüzgar yanılmışım ama sıcaklık yüksek daha çok basınc farkı oluşur

**Alıştırma 7 (`|` ve `&`):**
- Ölçüm **veya** GFS 20+ olan kaç saat var?
- İkisi **birden** 20+ olan kaç saat var? (GFS'in kuvvetli rüzgârı doğru yakaladığı saatler)

B1'deki 789 ve 49 sayılarıyla birlikte düşün: bu iki yeni sayı aralarında nasıl bir ilişkiyle bağlı?

In [13]:
veri[(veri["speed_kn_obs"]>=20)| (veri["speed_kn_gfs"]>=20)]
veri[(veri["speed_kn_obs"]>=20)& (veri["speed_kn_gfs"]>=20)]

,time,speed_kn_gfs,dir_deg_gfs,gust_kn_gfs,station,speed_kn_obs,dir_deg_obs,gust_kn_obs,hata
9195,2025-03-19 12:00:00+00:00,20.1,15,27.2,LTBJ,27.0,20.0,38.0,-6.9
9196,2025-03-19 13:00:00+00:00,21.7,15,28.2,LTBJ,26.0,10.0,39.0,-4.3
9197,2025-03-19 14:00:00+00:00,22.3,16,29.9,LTBJ,24.0,30.0,36.0,-1.7
10588,2025-05-16 13:00:00+00:00,21.4,176,36.0,LTBJ,20.0,190.0,NaN,1.4
10589,2025-05-16 14:00:00+00:00,20.9,175,36.0,LTBJ,21.0,190.0,31.0,-0.1
11641,2025-06-29 10:00:00+00:00,23.7,19,24.9,LTBJ,23.0,20.0,37.0,0.7
11642,2025-06-29 11:00:00+00:00,25.1,16,25.1,LTBJ,26.0,20.0,43.0,-0.9
11643,2025-06-29 12:00:00+00:00,22.4,14,24.7,LTBJ,27.0,20.0,NaN,-4.6
11644,2025-06-29 13:00:00+00:00,22.5,16,25.5,LTBJ,24.0,20.0,34.0,-1.5
12268,2025-07-25 13:00:00+00:00,20.1,17,20.2,LTBJ,20.0,20.0,NaN,0.1


> 789, 49 ve senin iki sayın arasındaki ilişki: (tek cümle, ipucu: küme kesişimi)
>
> *bneyi souyon anlaadım

**Alıştırma 8 (bonus, `.isna()` + `.loc`):** rüzgâr yönü (`dir_deg_obs`) boş olan saatlerde ölçülen rüzgâr hızının ortalaması kaç? Bütün saatlerin ortalamasıyla karşılaştır.

In [14]:
# Pezo: isna() bir sütunun METODU; önce sütunu seç, sonra sor
yon_bos = veri["dir_deg_obs"].isna()          # yön boş mu? -> maske
print("yön boş saat:", yon_bos.sum())
print("yön boşken ort. hız:", veri.loc[yon_bos, "speed_kn_obs"].mean())
print("bütün saatlerin ort. hızı:", veri["speed_kn_obs"].mean())

yön boş saat: 3399
yön boşken ort. hız: 2.060017652250662
bütün saatlerin ort. hızı: 9.103529210941835


> Yön neden tam bu saatlerde boş olabilir? (tahmin et; ipucu: rüzgâr çok hafifken yön ne olur?)
>
> *Pezo:* yön boşken ortalama hız ~2 knot, yani neredeyse rüzgâr yok. Rüzgâr çok hafifken yönü sürekli değişir; havalimanı raporu (METAR) bu durumda derece yerine "VRB" (değişken) ya da sakin yazar, bizim veride bu boş hücre olarak kalıyor. Yani bu boşluk bir hata değil, bir bilgi.

**Alıştırma 9 (bonus, `.dt.hour`):** **yerel saatle** 12:00–18:00 arasında kuvvetli saat oranı ne, günün geri kalanında ne?

İpucu: yerel saat = `(veri["time"].dt.hour + 3) % 24`. `%` bölümden kalan; 22+3=25 olunca 1'e döndürmek için.

In [15]:
# Pezo: saatler UTC, İzmir UTC+3. % 24: 22+3=25 olunca 1'e döndürür
yerel_saat = (veri["time"].dt.hour + 3) % 24
ogleden_sonra = (yerel_saat >= 12) & (yerel_saat < 18)   # 12:00-17:59

print("öğleden sonra kuvvetli oranı:", kuvvetli[ogleden_sonra].mean() * 100)
print("günün geri kalanı:           ", kuvvetli[~ogleden_sonra].mean() * 100)

öğleden sonra kuvvetli oranı: 5.582109935131444
günün geri kalanı:            2.6287339971550496


> Bu sonuç 3c'deki LSTM için neden önemli olabilir? (tek cümle)
>
> *Pezo:* kuvvetli rüzgâr öğleden sonra iki kattan fazla sık (deniz meltemi: kara ısınınca denizden rüzgâr çeker); yani "günün hangi saati" çok güçlü bir ipucu, model bunu görmezse kuvvetli saatleri kaçırır. Bizim modellerde saat zaten feature olarak var, bu sonuç onun neden işe yaradığını gösteriyor.